# Phase 3: LLM Integration in Tree of Thoughts

## Comprehensive Guide to Real API Integration

This notebook teaches you how to integrate real Large Language Models into your Tree of Thoughts implementations. We'll cover:
- API setup and authentication for Claude and OpenAI
- Prompting strategies optimized for ToT
- Building orchestrators for managing complex LLM workflows
- Complete working examples with real API patterns
- Production-ready best practices

**Note:** This notebook demonstrates patterns and approaches. Use with actual API keys in your own environment.

---
# Part 1: API Setup and Authentication

## 1.1 Environment Setup

First, install the required packages:

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'anthropic',      # For Claude API
    'openai>=1.0',    # For GPT-4 and GPT-3.5
    'python-dotenv',  # For environment variables
    'requests',       # For HTTP calls (fallback)
]

for package in packages:
    try:
        __import__(package.split('>=')[0].split('<')[0].replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

## 1.2 Environment Variables and Security Best Practices

Never hardcode API keys in notebooks. Use environment variables:

In [ ]:
import os
from dotenv import load_dotenv
import json
from typing import Optional

# Load environment variables from .env file
# Create a .env file in your project directory with:
# ANTHROPIC_API_KEY=your_key_here
# OPENAI_API_KEY=your_key_here

load_dotenv()

class APIConfig:
    """Secure configuration management for API credentials."""
    
    @staticmethod
    def get_anthropic_key() -> Optional[str]:
        """Get Anthropic API key from environment."""
        key = os.getenv('ANTHROPIC_API_KEY')
        if key and key.startswith('sk-ant-'):
            return key
        return None
    
    @staticmethod
    def get_openai_key() -> Optional[str]:
        """Get OpenAI API key from environment."""
        key = os.getenv('OPENAI_API_KEY')
        if key and key.startswith('sk-'):
            return key
        return None
    
    @staticmethod
    def mask_key(key: str, visible_chars: int = 4) -> str:
        """Safely display API key (for debugging only)."""
        if len(key) <= visible_chars:
            return '*' * len(key)
        return key[:visible_chars] + '*' * (len(key) - visible_chars)

print("✓ APIConfig loaded")
print(f"\nSetup Instructions:")
print("1. Create a .env file in your project root")
print("2. Add your keys:")
print("   ANTHROPIC_API_KEY=sk-ant-...")
print("   OPENAI_API_KEY=sk-...")
print("3. Never commit .env to version control")

## 1.3 Claude API Setup

Setting up the Anthropic Claude API:

In [ ]:
from anthropic import Anthropic, HUMAN_PROMPT, AI_PROMPT
import json

class ClaudeClient:
    """Wrapper for Claude API with caching and error handling."""
    
    def __init__(self, api_key: Optional[str] = None, model: str = "claude-3-5-sonnet-20241022"):
        self.api_key = api_key or APIConfig.get_anthropic_key()
        self.model = model
        self.client = Anthropic(api_key=self.api_key) if self.api_key else None
        self.total_tokens = 0
        self.total_cost = 0
        
        # Pricing per 1M tokens (as of 2024)
        self.pricing = {
            "claude-3-5-sonnet-20241022": {"input": 3, "output": 15},
            "claude-3-opus-20250219": {"input": 15, "output": 75},
            "claude-3-haiku-20240307": {"input": 0.80, "output": 4},
        }
    
    def _calculate_cost(self, input_tokens: int, output_tokens: int) -> float:
        """Calculate cost of API call in dollars."""
        pricing = self.pricing.get(self.model, {"input": 3, "output": 15})
        input_cost = (input_tokens / 1_000_000) * pricing["input"]
        output_cost = (output_tokens / 1_000_000) * pricing["output"]
        return input_cost + output_cost
    
    def generate(
        self,
        prompt: str,
        system_prompt: str = None,
        temperature: float = 0.7,
        max_tokens: int = 1024,
        stop_sequences: list = None,
    ) -> dict:
        """Generate text using Claude."""
        if not self.client:
            return self._mock_response(prompt, system_prompt)
        
        try:
            messages = [{"role": "user", "content": prompt}]
            kwargs = {
                "model": self.model,
                "max_tokens": max_tokens,
                "temperature": temperature,
                "messages": messages,
            }
            
            if system_prompt:
                kwargs["system"] = system_prompt
            
            if stop_sequences:
                kwargs["stop_sequences"] = stop_sequences
            
            response = self.client.messages.create(**kwargs)
            
            # Track token usage and cost
            input_tokens = response.usage.input_tokens
            output_tokens = response.usage.output_tokens
            cost = self._calculate_cost(input_tokens, output_tokens)
            
            self.total_tokens += input_tokens + output_tokens
            self.total_cost += cost
            
            return {
                "content": response.content[0].text,
                "tokens": {"input": input_tokens, "output": output_tokens},
                "cost": cost,
                "model": self.model,
                "stop_reason": response.stop_reason,
            }
        except Exception as e:
            print(f"Claude API Error: {str(e)}")
            return self._mock_response(prompt, system_prompt)
    
    def _mock_response(self, prompt: str, system_prompt: str) -> dict:
        """Return mock response when API key not available."""
        return {
            "content": "[Mock Claude response - no API key configured]",
            "tokens": {"input": 10, "output": 20},
            "cost": 0.0,
            "model": self.model,
            "stop_reason": "end_turn",
        }

# Initialize Claude client
claude = ClaudeClient(model="claude-3-5-sonnet-20241022")
print(f"✓ Claude client initialized (model: {claude.model})")

## 1.4 OpenAI API Setup

Setting up OpenAI's GPT models:

In [ ]:
from openai import OpenAI
from typing import List, Dict

class OpenAIClient:
    """Wrapper for OpenAI API with caching and error handling."""
    
    def __init__(self, api_key: Optional[str] = None, model: str = "gpt-4-turbo-preview"):
        self.api_key = api_key or APIConfig.get_openai_key()
        self.model = model
        self.client = OpenAI(api_key=self.api_key) if self.api_key else None
        self.total_tokens = 0
        self.total_cost = 0
        
        # Pricing per 1K tokens (as of 2024)
        self.pricing = {
            "gpt-4-turbo-preview": {"input": 0.01, "output": 0.03},
            "gpt-4": {"input": 0.03, "output": 0.06},
            "gpt-3.5-turbo": {"input": 0.0005, "output": 0.0015},
        }
    
    def _calculate_cost(self, input_tokens: int, output_tokens: int) -> float:
        """Calculate cost of API call in dollars."""
        pricing = self.pricing.get(self.model, {"input": 0.01, "output": 0.03})
        input_cost = (input_tokens / 1000) * pricing["input"]
        output_cost = (output_tokens / 1000) * pricing["output"]
        return input_cost + output_cost
    
    def generate(
        self,
        prompt: str,
        system_prompt: str = None,
        temperature: float = 0.7,
        max_tokens: int = 1024,
        top_p: float = 1.0,
    ) -> dict:
        """Generate text using OpenAI."""
        if not self.client:
            return self._mock_response(prompt, system_prompt)
        
        try:
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": prompt})
            
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                top_p=top_p,
            )
            
            # Track token usage and cost
            input_tokens = response.usage.prompt_tokens
            output_tokens = response.usage.completion_tokens
            cost = self._calculate_cost(input_tokens, output_tokens)
            
            self.total_tokens += input_tokens + output_tokens
            self.total_cost += cost
            
            return {
                "content": response.choices[0].message.content,
                "tokens": {"input": input_tokens, "output": output_tokens},
                "cost": cost,
                "model": self.model,
                "finish_reason": response.choices[0].finish_reason,
            }
        except Exception as e:
            print(f"OpenAI API Error: {str(e)}")
            return self._mock_response(prompt, system_prompt)
    
    def _mock_response(self, prompt: str, system_prompt: str) -> dict:
        """Return mock response when API key not available."""
        return {
            "content": "[Mock OpenAI response - no API key configured]",
            "tokens": {"input": 10, "output": 20},
            "cost": 0.0,
            "model": self.model,
            "finish_reason": "stop",
        }

# Initialize OpenAI client
openai_client = OpenAIClient(model="gpt-4-turbo-preview")
print(f"✓ OpenAI client initialized (model: {openai_client.model})")

---
# Part 2: Prompting Strategies for Tree of Thoughts

## 2.1 System Prompt Design for Thought Generation

Good system prompts are critical for ToT performance:

In [ ]:
class PromptLibrary:
    """Collection of domain-specific prompts for Tree of Thoughts."""
    
    # Generic generation prompt
    GENERIC_GENERATION = """You are an expert problem solver. Your task is to propose a next step or partial solution to the following problem.

{problem}

Previous steps taken:
{previous_steps}

Propose a single concrete next step or reasoning approach. Be specific and actionable."""
    
    # Math problem solving
    MATH_GENERATION = """You are a mathematical problem solver. You excel at breaking down complex problems into clear steps.

Problem: {problem}

Previous reasoning:
{previous_steps}

Propose the next logical mathematical step. Consider:
- Relevant formulas or theorems
- Simplification opportunities
- Alternative approaches

Output ONLY the next step, no explanation."""
    
    # Creative writing
    WRITING_GENERATION = """You are a creative writing expert. Your task is to generate diverse narrative directions.

Prompt: {problem}

Existing narrative:
{previous_steps}

Suggest ONE creative continuation that:
- Builds on the existing narrative
- Introduces interesting conflict or development
- Is distinct from typical approaches

Keep it concise (1-3 sentences)."""
    
    # Code debugging
    CODE_GENERATION = """You are an expert programmer. Your task is to debug and improve code.

Code with issue:
{problem}

Steps taken so far:
{previous_steps}

Suggest ONE specific debugging or improvement step:
- Identify a potential issue
- Propose a fix or test
- Consider edge cases

Be concrete and testable."""
    
    # Generic evaluation
    GENERIC_EVALUATION = """You are an expert evaluator. Rate the quality of this solution step.

Problem: {problem}

Proposed step: {candidate}

Rate this step on a scale of 0-10 where:
- 0-2: Invalid, incorrect, or harmful
- 3-4: Poor, unlikely to help
- 5-6: Mediocre, has some merit
- 7-8: Good, likely helpful
- 9-10: Excellent, strong progress

Respond with ONLY a JSON object: {"score": <number>, "reasoning": "<brief explanation>"}"""
    
    # Math evaluation
    MATH_EVALUATION = """You are a mathematics expert evaluator.

Problem: {problem}

Proposed step: {candidate}

Evaluate this step:
1. Is the mathematics correct?
2. Does it make progress toward the solution?
3. Are there any logical gaps?

Score 0-10 and provide reasoning.
Respond with JSON: {"score": <number>, "reasoning": "<brief explanation>"}"""
    
    # Writing evaluation
    WRITING_EVALUATION = """You are a literary critic.

Writing prompt: {problem}

Proposed continuation: {candidate}

Evaluate on:
1. Creativity and originality
2. Narrative coherence
3. Writing quality
4. Interest level

Score 0-10 where 10 is exceptional.
Respond with JSON: {"score": <number>, "reasoning": "<brief explanation>"}"""
    
    @staticmethod
    def get_generation_prompt(domain: str = "generic") -> str:
        """Get generation prompt for domain."""
        prompts = {
            "math": PromptLibrary.MATH_GENERATION,
            "writing": PromptLibrary.WRITING_GENERATION,
            "code": PromptLibrary.CODE_GENERATION,
            "generic": PromptLibrary.GENERIC_GENERATION,
        }
        return prompts.get(domain, PromptLibrary.GENERIC_GENERATION)
    
    @staticmethod
    def get_evaluation_prompt(domain: str = "generic") -> str:
        """Get evaluation prompt for domain."""
        prompts = {
            "math": PromptLibrary.MATH_EVALUATION,
            "writing": PromptLibrary.WRITING_EVALUATION,
            "generic": PromptLibrary.GENERIC_EVALUATION,
        }
        return prompts.get(domain, PromptLibrary.GENERIC_EVALUATION)

print("✓ PromptLibrary loaded with 7 domain-specific prompts")

## 2.2 Temperature and Sampling Parameters

Understanding how to control diversity in model outputs:

In [ ]:
class SamplingConfig:
    """Best practices for temperature and sampling parameters in ToT."""
    
    # For generation: we want DIVERSITY
    # This helps explore more branches of the thought tree
    GENERATION_CONFIGS = {
        "diverse": {  # For broad exploration
            "temperature": 0.9,
            "top_p": 0.95,
            "description": "High diversity - explores many branches"
        },
        "balanced": {  # Default - good balance
            "temperature": 0.7,
            "top_p": 0.85,
            "description": "Balanced exploration and quality"
        },
        "focused": {  # Lower diversity
            "temperature": 0.5,
            "top_p": 0.9,
            "description": "Focus on high-quality candidates"
        },
    }
    
    # For evaluation: we want CONSISTENCY
    # This ensures we get reliable scoring
    EVALUATION_CONFIGS = {
        "strict": {  # Maximum consistency
            "temperature": 0.0,
            "top_p": 1.0,
            "description": "Deterministic - same input = same score"
        },
        "stable": {  # Good consistency with slight variation
            "temperature": 0.2,
            "top_p": 1.0,
            "description": "Stable scoring with minor variation"
        },
    }
    
    @staticmethod
    def get_generation_config(mode: str = "balanced") -> dict:
        """Get generation sampling config."""
        config = SamplingConfig.GENERATION_CONFIGS.get(mode, SamplingConfig.GENERATION_CONFIGS["balanced"])
        return {k: v for k, v in config.items() if k != "description"}
    
    @staticmethod
    def get_evaluation_config(mode: str = "stable") -> dict:
        """Get evaluation sampling config."""
        config = SamplingConfig.EVALUATION_CONFIGS.get(mode, SamplingConfig.EVALUATION_CONFIGS["stable"])
        return {k: v for k, v in config.items() if k != "description"}

print("✓ SamplingConfig loaded")
print("\nGeneration Configs:")
for name, config in SamplingConfig.GENERATION_CONFIGS.items():
    print(f"  {name}: T={config['temperature']} P={config['top_p']} - {config['description']}")
print("\nEvaluation Configs:")
for name, config in SamplingConfig.EVALUATION_CONFIGS.items():
    print(f"  {name}: T={config['temperature']} P={config['top_p']} - {config['description']}")

## 2.3 Cost Optimization Strategies

Key techniques to reduce API costs while maintaining quality:

In [ ]:
class CostOptimization:
    """Strategies for optimizing costs in LLM-based ToT."""
    
    @staticmethod
    def estimate_cost(
        num_candidates: int,
        branches: int,
        depth: int,
        use_cheaper_model_for_eval: bool = False,
    ) -> dict:
        """Estimate cost of a ToT search.
        
        Args:
            num_candidates: Number of candidates per step
            branches: Number of branches to explore (kept)
            depth: Depth of tree
            use_cheaper_model_for_eval: Use GPT-3.5 instead of GPT-4 for evaluation
        """
        total_generations = num_candidates * (branches ** (depth - 1))
        total_evaluations = num_candidates * (branches ** (depth - 1))
        
        # Pricing (per 1M tokens for Claude, per 1K for OpenAI)
        generation_cost_gpt4 = (total_generations * 100) / 1000 * 0.01  # ~100 tokens per generation
        evaluation_cost_gpt4 = (total_evaluations * 50) / 1000 * 0.01   # ~50 tokens per evaluation
        evaluation_cost_gpt35 = (total_evaluations * 50) / 1000 * 0.0005 # 10x cheaper
        
        total_gpt4 = generation_cost_gpt4 + evaluation_cost_gpt4
        total_mixed = generation_cost_gpt4 + (evaluation_cost_gpt35 if use_cheaper_model_for_eval else evaluation_cost_gpt4)
        
        return {
            "total_generations": total_generations,
            "total_evaluations": total_evaluations,
            "cost_gpt4_only": round(total_gpt4, 2),
            "cost_mixed": round(total_mixed, 2),
            "savings_with_mixed": round(total_gpt4 - total_mixed, 2),
        }
    
    @staticmethod
    def optimization_strategies() -> dict:
        """Key strategies for cost optimization."""
        return {
            "Use cheaper models for evaluation": {
                "description": "GPT-3.5 for evaluation, GPT-4 for generation",
                "savings": "~90% on evaluation costs",
                "tradeoff": "Slightly lower quality evaluation"
            },
            "Reduce breadth not depth": {
                "description": "Fewer candidates per step (2-3 vs 5-10)",
                "savings": "Linear reduction in cost",
                "tradeoff": "May miss good branches"
            },
            "Caching and memoization": {
                "description": "Cache results for identical prompts",
                "savings": "10-50% depending on problem structure",
                "tradeoff": "Requires exact matching of prompts"
            },
            "Batch processing": {
                "description": "Process multiple candidates in single call",
                "savings": "5-15% on overhead",
                "tradeoff": "More complex prompt engineering"
            },
            "Early pruning": {
                "description": "Eliminate clearly bad candidates early",
                "savings": "20-40% by cutting bad branches",
                "tradeoff": "Risk of pruning good branches"
            },
        }

print("✓ CostOptimization loaded")
print("\nExample Cost Estimates:")
print(json.dumps(CostOptimization.estimate_cost(5, 2, 3), indent=2))

---
# Part 3: Building Generation and Evaluation Prompts

## 3.1 Prompt Template System

A flexible system for building and managing prompts:

In [ ]:
class PromptTemplate:
    """Flexible prompt template system with validation."""
    
    def __init__(self, template: str, required_vars: list = None):
        self.template = template
        self.required_vars = required_vars or self._extract_vars(template)
    
    @staticmethod
    def _extract_vars(template: str) -> list:
        """Extract variable names from template."""
        import re
        return re.findall(r'\{(\w+)\}', template)
    
    def format(self, **kwargs) -> str:
        """Format template with values."""
        missing = set(self.required_vars) - set(kwargs.keys())
        if missing:
            raise ValueError(f"Missing required variables: {missing}")
        return self.template.format(**kwargs)
    
    def __repr__(self) -> str:
        return f"PromptTemplate(vars={self.required_vars})"

# Create domain-specific prompt templates
generation_template = PromptTemplate(
    """You are solving a {domain} problem using Tree of Thoughts.

Problem: {problem}

Current solution path:
{current_path}

Generate ONE diverse next step that:
1. Is different from previous attempts
2. Makes logical progress
3. Is testable/verifiable

Output ONLY the next step.""",
    required_vars=["domain", "problem", "current_path"]
)

evaluation_template = PromptTemplate(
    """You are evaluating solution steps for a {domain} problem.

Problem: {problem}

Proposed step: {candidate}

Evaluate on:
1. Correctness: Is it factually/logically correct?
2. Progress: Does it move toward solving the problem?
3. Clarity: Is it clear and actionable?

Score 0-10 where:
- 0-2: Invalid or harmful
- 3-4: Poor quality
- 5-6: Acceptable
- 7-8: Good
- 9-10: Excellent

Respond with JSON only: {"score": <int>, "reason": "<brief>"}.""",
    required_vars=["domain", "problem", "candidate"]
)

print("✓ PromptTemplate system created")
print(f"\nGeneration template requires: {generation_template.required_vars}")
print(f"Evaluation template requires: {evaluation_template.required_vars}")

## 3.2 Structured Output Parsing

Reliably extract structured data from LLM outputs:

In [ ]:
import json
import re
from typing import Dict, Any

class OutputParser:
    """Parse structured output from LLMs."""
    
    @staticmethod
    def extract_json(text: str) -> Dict[str, Any]:
        """Extract JSON from text, even if not perfectly formatted."""
        # Try direct JSON parse first
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass
        
        # Try extracting JSON block
        json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group())
            except json.JSONDecodeError:
                pass
        
        # Fallback: try to extract score
        score_match = re.search(r'["\']?score["\']?\s*:\s*(\d+)', text, re.IGNORECASE)
        if score_match:
            return {"score": int(score_match.group(1)), "reason": "Extracted from text"}
        
        return {"score": 5, "reason": "Failed to parse output"}
    
    @staticmethod
    def extract_score(text: str) -> float:
        """Extract numerical score from evaluation."""
        parsed = OutputParser.extract_json(text)
        return float(parsed.get("score", 5))
    
    @staticmethod
    def validate_evaluation(result: dict) -> bool:
        """Validate evaluation result structure."""
        return (
            "score" in result and
            isinstance(result["score"], (int, float)) and
            0 <= result["score"] <= 10
        )

print("✓ OutputParser loaded")

# Test parsing
test_output = '{"score": 7, "reason": "Good progress"}'
parsed = OutputParser.extract_json(test_output)
print(f"\nTest parse: {parsed}")
print(f"Valid: {OutputParser.validate_evaluation(parsed)}")

---
# Part 4: Complete TreeOfThoughtsOrchestrator Class

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Callable
import time

@dataclass
class ThoughtNode:
    """A node in the tree of thoughts."""
    content: str
    depth: int = 0
    parent: Optional['ThoughtNode'] = None
    score: Optional[float] = None
    children: List['ThoughtNode'] = field(default_factory=list)
    tokens_used: Dict[str, int] = field(default_factory=lambda: {"input": 0, "output": 0})
    cost: float = 0.0
    
    def get_path(self) -> List[str]:
        """Get path from root to this node."""
        path = []
        node = self
        while node:
            path.insert(0, node.content)
            node = node.parent
        return path
    
    def __repr__(self) -> str:
        return f"ThoughtNode(depth={self.depth}, score={self.score})"


class TreeOfThoughtsOrchestrator:
    """Complete orchestrator for Tree of Thoughts with real LLMs."""
    
    def __init__(
        self,
        problem: str,
        llm_client: 'ClaudeClient' | 'OpenAIClient' = None,
        evaluator_client: 'ClaudeClient' | 'OpenAIClient' = None,
        domain: str = "generic",
        max_depth: int = 5,
        num_candidates: int = 3,
        num_branches: int = 2,
    ):
        self.problem = problem
        self.llm_client = llm_client or claude
        self.evaluator_client = evaluator_client or self.llm_client
        self.domain = domain
        self.max_depth = max_depth
        self.num_candidates = num_candidates
        self.num_branches = num_branches
        
        # Tree state
        self.root = ThoughtNode("", depth=0)
        self.frontier = [self.root]  # Active nodes to expand
        
        # Tracking
        self.total_calls = 0
        self.total_tokens = 0
        self.total_cost = 0
        self.all_nodes = [self.root]
    
    def _generate_candidates(self, parent_node: ThoughtNode) -> List[ThoughtNode]:
        """Generate candidate next steps."""
        candidates = []
        current_path = "\n".join(parent_node.get_path())
        
        for i in range(self.num_candidates):
            prompt = generation_template.format(
                domain=self.domain,
                problem=self.problem,
                current_path=current_path or "[Starting from scratch]"
            )
            
            # Add some variation on sampling config
            sampling = SamplingConfig.get_generation_config("balanced")
            
            response = self.llm_client.generate(
                prompt=prompt,
                system_prompt=f"You are a {self.domain} expert helping solve problems.",
                temperature=sampling["temperature"],
                max_tokens=256,
            )
            
            # Create node
            node = ThoughtNode(
                content=response["content"],
                depth=parent_node.depth + 1,
                parent=parent_node,
                tokens_used=response["tokens"],
                cost=response["cost"],
            )
            
            candidates.append(node)
            self.all_nodes.append(node)
            self.total_calls += 1
            self.total_tokens += response["tokens"]["input"] + response["tokens"]["output"]
            self.total_cost += response["cost"]
        
        return candidates
    
    def _evaluate_candidates(self, candidates: List[ThoughtNode]) -> List[ThoughtNode]:
        """Evaluate candidates and return top-k."""
        for candidate in candidates:
            prompt = evaluation_template.format(
                domain=self.domain,
                problem=self.problem,
                candidate=candidate.content
            )
            
            sampling = SamplingConfig.get_evaluation_config("stable")
            
            response = self.evaluator_client.generate(
                prompt=prompt,
                system_prompt=f"You are an expert evaluator for {self.domain} problems.",
                temperature=sampling["temperature"],
                max_tokens=100,
            )
            
            # Parse score
            parsed = OutputParser.extract_json(response["content"])
            score = parsed.get("score", 5)
            candidate.score = float(score)
            
            self.total_calls += 1
            self.total_tokens += response["tokens"]["input"] + response["tokens"]["output"]
            self.total_cost += response["cost"]
        
        # Sort by score and keep top-k
        candidates.sort(key=lambda x: x.score, reverse=True)
        return candidates[:self.num_branches]
    
    def search(self, verbose: bool = True) -> dict:
        """Execute tree of thoughts search."""
        if verbose:
            print(f"Starting ToT search: depth={self.max_depth}, candidates={self.num_candidates}, branches={self.num_branches}")
        
        start_time = time.time()
        
        for depth in range(self.max_depth):
            if verbose:
                print(f"\n[Depth {depth}] Processing {len(self.frontier)} nodes...")
            
            new_frontier = []
            
            for parent in self.frontier:
                # Generate candidates
                candidates = self._generate_candidates(parent)
                
                # Evaluate and select
                selected = self._evaluate_candidates(candidates)
                
                # Add to parent and frontier
                parent.children = selected
                new_frontier.extend(selected)
                
                if verbose:
                    avg_score = sum(c.score for c in selected) / len(selected)
                    print(f"  Generated {len(candidates)}, selected {len(selected)} (avg score: {avg_score:.1f})")
            
            self.frontier = new_frontier
        
        elapsed = time.time() - start_time
        
        # Find best path
        best_leaf = max(self.frontier, key=lambda x: x.score)
        best_path = best_leaf.get_path()
        
        return {
            "best_path": best_path,
            "best_score": best_leaf.score,
            "total_nodes": len(self.all_nodes),
            "total_calls": self.total_calls,
            "total_tokens": self.total_tokens,
            "total_cost": round(self.total_cost, 4),
            "elapsed_seconds": round(elapsed, 2),
        }

print("✓ TreeOfThoughtsOrchestrator loaded")

---
# Part 5: Complete Working Examples

## Example 1: Math Problem Solving with Claude API

Solving the Game of 24 using Tree of Thoughts:

In [ ]:
# Example 1: Game of 24 with Claude
print("="*60)
print("EXAMPLE 1: Game of 24 with Claude API")
print("="*60)

math_problem = """Find a way to combine the numbers 3, 8, 3, 8 using +, -, *, / to get 24.
Each number must be used exactly once."""

print(f"Problem: {math_problem}")
print(f"\nInitializing ToT orchestrator...")

# Create orchestrator with Claude
orchestrator = TreeOfThoughtsOrchestrator(
    problem=math_problem,
    llm_client=claude,
    evaluator_client=claude,
    domain="math",
    max_depth=2,  # Keep shallow for demo
    num_candidates=3,
    num_branches=2,
)

print(f"Starting search...")
result = orchestrator.search(verbose=True)

print(f"\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"Best path found:")
for i, step in enumerate(result['best_path']):
    if step:  # Skip empty root
        print(f"  Step {i}: {step}")
print(f"\nBest score: {result['best_score']:.1f}/10")
print(f"Total LLM calls: {result['total_calls']}")
print(f"Tokens used: {result['total_tokens']}")
print(f"Cost: ${result['total_cost']}")
print(f"Time elapsed: {result['elapsed_seconds']}s")

## Example 2: Creative Writing with OpenAI API

In [ ]:
# Example 2: Creative writing with OpenAI
print("\n" + "="*60)
print("EXAMPLE 2: Creative Writing with OpenAI API")
print("="*60)

writing_problem = """Write an opening paragraph for a sci-fi story that begins with:
'The station's power went out at midnight.'
Make it compelling and introduce a sense of mystery."""

print(f"Problem: {writing_problem}")
print(f"\nInitializing ToT orchestrator...")

# Create orchestrator with OpenAI
orchestrator = TreeOfThoughtsOrchestrator(
    problem=writing_problem,
    llm_client=openai_client,
    evaluator_client=openai_client,
    domain="writing",
    max_depth=2,
    num_candidates=3,
    num_branches=2,
)

print(f"Starting search...")
result = orchestrator.search(verbose=True)

print(f"\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"Best narrative path:")
for i, step in enumerate(result['best_path']):
    if step:
        print(f"\n[Step {i}]")
        print(step[:200] + "..." if len(step) > 200 else step)
print(f"\nBest score: {result['best_score']:.1f}/10")
print(f"Total LLM calls: {result['total_calls']}")
print(f"Tokens used: {result['total_tokens']}")
print(f"Cost: ${result['total_cost']}")
print(f"Time elapsed: {result['elapsed_seconds']}s")

## Example 3: Code Debugging with Both APIs

Demonstrate using Claude for generation and OpenAI for evaluation:

In [ ]:
# Example 3: Code debugging - mixed API approach
print("\n" + "="*60)
print("EXAMPLE 3: Code Debugging (Claude generation + OpenAI evaluation)")
print("="*60)

code_problem = """This Python function has a bug:

```python
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)
```

The problem: it's extremely slow for large n due to redundant calculations.
Propose a fix step by step."""

print(f"Problem: Code is inefficient due to repeated calculations")
print(f"\nInitializing ToT orchestrator with mixed APIs...")

# Use Claude for generation (better at code), OpenAI for evaluation
orchestrator = TreeOfThoughtsOrchestrator(
    problem=code_problem,
    llm_client=claude,  # Claude for generation
    evaluator_client=openai_client,  # OpenAI for evaluation
    domain="code",
    max_depth=2,
    num_candidates=3,
    num_branches=2,
)

print(f"Starting search...")
result = orchestrator.search(verbose=True)

print(f"\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"Best debugging approach:")
for i, step in enumerate(result['best_path']):
    if step:
        print(f"\n[Step {i}]")
        print(step[:250] + "..." if len(step) > 250 else step)
print(f"\nBest score: {result['best_score']:.1f}/10")
print(f"Total LLM calls: {result['total_calls']}")
print(f"Tokens used: {result['total_tokens']}")
print(f"Cost: ${result['total_cost']}")
print(f"\nNote: This demo uses Claude for generation, OpenAI for evaluation")

---
# Part 6: Best Practices

## 6.1 Batch Processing vs Sequential Calls

In [ ]:
class BatchProcessor:
    """Demonstrates batch processing strategies for ToT."""
    
    @staticmethod
    def sequential_approach() -> dict:
        """Traditional: one API call per candidate."""
        return {
            "description": "Single prompt, single response per candidate",
            "advantages": [
                "Simple to implement",
                "Clear error handling per candidate",
                "Can handle variable length responses",
            ],
            "disadvantages": [
                "Higher API call overhead",
                "No parallelization benefit",
                "Slower overall execution",
            ],
            "best_for": "Quick experiments, small ToT trees",
        }
    
    @staticmethod
    def batch_approach() -> dict:
        """Advanced: multiple candidates in single prompt."""
        return {
            "description": "One prompt generates multiple candidates at once",
            "advantages": [
                "50% fewer API calls",
                "Potential cost savings",
                "Better context awareness between candidates",
            ],
            "disadvantages": [
                "Complex prompt engineering",
                "Harder to parse multiple outputs",
                "Risk of quality degradation",
            ],
            "best_for": "Production systems, cost-sensitive applications",
            "example_prompt": """Generate 3 diverse solutions to: {problem}

Format your response as:
Candidiate 1: ...
Candidate 2: ...
Candidate 3: ...""",
        }
    
    @staticmethod
    def async_batch_approach() -> dict:
        """Hybrid: parallel API calls for multiple candidates."""
        return {
            "description": "Send multiple requests concurrently",
            "advantages": [
                "Maintains sequential quality",
                "Reduced wall-clock time",
                "Better resource utilization",
            ],
            "disadvantages": [
                "Requires async/await support",
                "Risk of rate limiting",
                "Harder debugging",
            ],
            "best_for": "High-throughput systems, rate limits allow it",
        }

print("✓ Batch processing strategies:")
for name, strategy in [
    ("Sequential", BatchProcessor.sequential_approach()),
    ("Batch", BatchProcessor.batch_approach()),
    ("Async Batch", BatchProcessor.async_batch_approach()),
]:
    print(f"\n{name}:")
    print(f"  {strategy['description']}")
    print(f"  Best for: {strategy['best_for']}")

## 6.2 Caching and Memoization

In [ ]:
import hashlib
from functools import lru_cache

class PromptCache:
    """Caching system for LLM responses to avoid redundant calls."""
    
    def __init__(self):
        self.cache = {}
        self.hits = 0
        self.misses = 0
    
    def _hash_prompt(self, prompt: str, model: str) -> str:
        """Create unique key for prompt+model."""
        combined = f"{model}:{prompt}"
        return hashlib.sha256(combined.encode()).hexdigest()[:16]
    
    def get(self, prompt: str, model: str) -> Optional[dict]:
        """Retrieve cached response if available."""
        key = self._hash_prompt(prompt, model)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None
    
    def set(self, prompt: str, model: str, response: dict) -> None:
        """Cache a response."""
        key = self._hash_prompt(prompt, model)
        self.cache[key] = response
    
    def hit_rate(self) -> float:
        """Calculate cache hit rate."""
        total = self.hits + self.misses
        return (self.hits / total * 100) if total > 0 else 0

# Example: Integrate with LLM client
class CachedClaudeClient(ClaudeClient):
    """Claude client with built-in caching."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.cache = PromptCache()
    
    def generate(self, prompt: str, **kwargs) -> dict:
        """Generate with automatic caching."""
        # Check cache first
        cached = self.cache.get(prompt, self.model)
        if cached:
            return cached
        
        # Cache miss - call API
        result = super().generate(prompt, **kwargs)
        self.cache.set(prompt, self.model, result)
        return result

print("✓ Caching system loaded")
print("\nKey benefits of caching:")
print("  - 100% cost savings for cache hits")
print("  - No quality loss (exact same response)")
print("  - Faster execution (no API latency)")

## 6.3 Token Budget Management

In [ ]:
class TokenBudget:
    """Manage token and cost budgets for ToT operations."""
    
    def __init__(
        self,
        max_tokens: Optional[int] = None,
        max_cost: Optional[float] = None,
        max_calls: Optional[int] = None,
    ):
        self.max_tokens = max_tokens
        self.max_cost = max_cost
        self.max_calls = max_calls
        
        self.tokens_used = 0
        self.cost_used = 0
        self.calls_made = 0
    
    def can_use_tokens(self, tokens: int) -> bool:
        """Check if we can use additional tokens."""
        if self.max_tokens is None:
            return True
        return self.tokens_used + tokens <= self.max_tokens
    
    def can_use_cost(self, cost: float) -> bool:
        """Check if we can use additional cost."""
        if self.max_cost is None:
            return True
        return self.cost_used + cost <= self.max_cost
    
    def can_make_call(self) -> bool:
        """Check if we can make another call."""
        if self.max_calls is None:
            return True
        return self.calls_made + 1 <= self.max_calls
    
    def add_call(self, tokens: int, cost: float) -> bool:
        """Record a call. Returns False if budget exceeded."""
        if not (self.can_use_tokens(tokens) and self.can_use_cost(cost) and self.can_make_call()):
            return False
        
        self.tokens_used += tokens
        self.cost_used += cost
        self.calls_made += 1
        return True
    
    def utilization(self) -> dict:
        """Get budget utilization percentages."""
        return {
            "tokens": (self.tokens_used / self.max_tokens * 100) if self.max_tokens else None,
            "cost": (self.cost_used / self.max_cost * 100) if self.max_cost else None,
            "calls": (self.calls_made / self.max_calls * 100) if self.max_calls else None,
        }

# Example usage
budget = TokenBudget(
    max_tokens=10000,
    max_cost=1.0,  # $1 max
    max_calls=100,
)

print("✓ TokenBudget system loaded")
print(f"\nExample budget constraints:")
print(f"  Max tokens: {budget.max_tokens}")
print(f"  Max cost: ${budget.max_cost}")
print(f"  Max calls: {budget.max_calls}")

## 6.4 Prompt Optimization Techniques

In [ ]:
class PromptOptimizer:
    """Techniques for optimizing prompts for better LLM performance."""
    
    @staticmethod
    def few_shot_examples(task: str) -> str:
        """Add few-shot examples to prompt."""
        return f"""Your task: {task}

Examples of good outputs:

Example 1:
Input: Simple problem
Output: Clear, step-by-step solution

Example 2:
Input: Complex problem
Output: Breaks down into sub-problems

Now, solve the actual problem..."""
    
    @staticmethod
    def chain_of_thought(task: str) -> str:
        """Encourage step-by-step reasoning."""
        return f"""Task: {task}

Let's think through this step by step:
1. First, understand the problem
2. Then, identify key constraints
3. Next, explore possible approaches
4. Finally, select the best approach

Go ahead..."""
    
    @staticmethod
    def role_based_prompt(role: str, task: str) -> str:
        """Assign a specific role to improve performance."""
        return f"""You are a {role}. Your expertise is in this field.

Task: {task}

Using your knowledge as a {role}, solve this..."""
    
    @staticmethod
    def structured_output(task: str, output_format: str) -> str:
        """Request structured output."""
        return f"""Task: {task}

Provide your response in this format:
{output_format}

Ensure your response follows this structure exactly."""
    
    @staticmethod
    def optimization_tips() -> dict:
        return {
            "Be specific": "Instead of 'analyze this', say 'identify 3 strengths and 2 weaknesses'",
            "Use examples": "Few-shot examples improve accuracy by 10-50%",
            "Clear format": "Specify output format (JSON, numbered list, etc)",
            "Role assignment": "'You are a scientist' works better than no role",
            "Explicit reasoning": "'Think step by step' improves complex problem solving",
            "Constraint clarity": "Clearly state what NOT to do",
            "Length guidance": "Specify desired response length (brief, detailed, etc)",
        }

print("✓ PromptOptimizer loaded")
print("\nKey optimization techniques:")
for technique, tip in list(PromptOptimizer.optimization_tips().items())[:4]:
    print(f"  {technique}: {tip}")

---
# Part 7: Production Considerations

## 7.1 Rate Limiting and Quota Management

In [ ]:
import time
from collections import deque

class RateLimiter:
    """Rate limiting for API calls to respect quotas."""
    
    def __init__(self, calls_per_minute: int = 60):
        self.calls_per_minute = calls_per_minute
        self.call_times = deque(maxlen=calls_per_minute)
    
    def wait_if_needed(self) -> float:
        """Wait if necessary to respect rate limit."""
        now = time.time()
        
        # Remove calls older than 1 minute
        while self.call_times and self.call_times[0] < now - 60:
            self.call_times.popleft()
        
        # If at capacity, wait
        if len(self.call_times) >= self.calls_per_minute:
            wait_time = 60 - (now - self.call_times[0])
            if wait_time > 0:
                time.sleep(wait_time)
                now = time.time()
        
        self.call_times.append(now)
        return now

class QuotaManager:
    """Track and manage API quotas."""
    
    def __init__(self):
        self.quotas = {}
        self.usage = {}
    
    def set_quota(self, api: str, limit: int, window_minutes: int = 60):
        """Set quota for an API."""
        self.quotas[api] = {"limit": limit, "window": window_minutes}
        self.usage[api] = deque()
    
    def check_quota(self, api: str) -> bool:
        """Check if quota available."""
        if api not in self.quotas:
            return True  # No quota set
        
        quota = self.quotas[api]
        now = time.time()
        window = quota["window"] * 60
        
        # Remove old entries
        while self.usage[api] and self.usage[api][0] < now - window:
            self.usage[api].popleft()
        
        return len(self.usage[api]) < quota["limit"]
    
    def record_usage(self, api: str):
        """Record an API usage."""
        if api in self.usage:
            self.usage[api].append(time.time())

print("✓ Rate limiting and quota management loaded")
print("\nExample quota setup:")
qm = QuotaManager()
qm.set_quota("claude", 100, window_minutes=60)
qm.set_quota("openai", 3000, window_minutes=60)
print("  Claude: 100 calls/hour")
print("  OpenAI: 3000 calls/hour")

## 7.2 Fallback Strategies

In [ ]:
class FallbackStrategy:
    """Fallback strategies when primary API fails."""
    
    @staticmethod
    def cascading_fallback(clients: List[dict]) -> dict:
        """Try multiple clients in order."""
        return {
            "strategy": "Cascading",
            "description": "Try primary, then fallback, then fallback2",
            "example": [
                {"provider": "claude", "model": "claude-3-5-sonnet"},
                {"provider": "openai", "model": "gpt-4-turbo"},
                {"provider": "openai", "model": "gpt-3.5-turbo"},
            ],
            "advantages": ["High availability", "Graceful degradation"],
            "disadvantages": ["Different quality levels"],
        }
    
    @staticmethod
    def timeout_retry_strategy() -> dict:
        """Retry with exponential backoff."""
        return {
            "strategy": "Timeout with Exponential Backoff",
            "description": "Retry failed calls with increasing delays",
            "parameters": {
                "initial_wait": "1 second",
                "max_wait": "60 seconds",
                "multiplier": 2,
                "max_retries": 3,
            },
            "code_example": """for attempt in range(max_retries):
    try:
        return llm_call()
    except APIError:
        wait = initial_wait * (multiplier ** attempt)
        time.sleep(wait)""",
        }
    
    @staticmethod
    def degraded_mode_strategy() -> dict:
        """Reduce tree complexity when under load."""
        return {
            "strategy": "Degraded Mode",
            "description": "Reduce ToT complexity when APIs fail",
            "parameters": {
                "normal_mode": {"candidates": 5, "depth": 4},
                "degraded_mode": {"candidates": 2, "depth": 2},
                "cache_hits_threshold": "50%",
            },
            "benefits": ["Stay operational", "Reduce costs", "Faster response"],
        }

print("✓ Fallback strategies loaded")
print("\nThree key strategies:")
for i, strategy in enumerate([
    FallbackStrategy.cascading_fallback([]),
    FallbackStrategy.timeout_retry_strategy(),
    FallbackStrategy.degraded_mode_strategy(),
], 1):
    print(f"\n{i}. {strategy['strategy']}")
    print(f"   {strategy['description']}")

## 7.3 Cost Monitoring and Optimization

In [ ]:
class CostMonitor:
    """Monitor and analyze costs in real-time."""
    
    def __init__(self):
        self.calls = []  # List of {timestamp, api, cost, tokens}
        self.total_cost = 0
        self.start_time = time.time()
    
    def record_call(self, api: str, cost: float, tokens: int):
        """Record an API call."""
        self.calls.append({
            "timestamp": time.time(),
            "api": api,
            "cost": cost,
            "tokens": tokens,
        })
        self.total_cost += cost
    
    def cost_by_api(self) -> dict:
        """Breakdown cost by API."""
        by_api = {}
        for call in self.calls:
            api = call["api"]
            if api not in by_api:
                by_api[api] = {"cost": 0, "calls": 0}
            by_api[api]["cost"] += call["cost"]
            by_api[api]["calls"] += 1
        return by_api
    
    def cost_per_token(self) -> dict:
        """Calculate efficiency metrics."""
        total_tokens = sum(c["tokens"] for c in self.calls)
        return {
            "total_tokens": total_tokens,
            "total_cost": self.total_cost,
            "cost_per_1k_tokens": (self.total_cost / total_tokens * 1000) if total_tokens > 0 else 0,
            "efficiency": "Good" if (self.total_cost / max(total_tokens, 1) * 1000) < 0.05 else "Poor",
        }
    
    def optimization_opportunities(self) -> List[str]:
        """Identify cost optimization opportunities."""
        opportunities = []
        efficiency = self.cost_per_token()
        
        if efficiency["cost_per_1k_tokens"] > 0.05:
            opportunities.append("Consider using cheaper models")
        
        by_api = self.cost_by_api()
        if "openai" in by_api and by_api["openai"]["cost"] > self.total_cost * 0.8:
            opportunities.append("Heavy OpenAI usage - consider Claude for generation")
        
        if len(self.calls) > 100:
            opportunities.append("Many calls - implement caching")
        
        return opportunities if opportunities else ["Cost optimization profile looks good!"]

print("✓ CostMonitor loaded")

# Example
monitor = CostMonitor()
monitor.record_call("claude", 0.05, 100)
monitor.record_call("openai", 0.10, 200)
print(f"\nExample cost tracking:")
print(f"  By API: {monitor.cost_by_api()}")
print(f"  Efficiency: {monitor.cost_per_token()}")

## 7.4 Monitoring and Logging Patterns

In [ ]:
import logging
from datetime import datetime

class ProductionLogger:
    """Logging system for production ToT operations."""
    
    def __init__(self, name: str = "TreeOfThoughts"):
        self.logger = logging.getLogger(name)
        self.logger.setLevel(logging.INFO)
        
        # Console handler
        ch = logging.StreamHandler()
        formatter = logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        ch.setFormatter(formatter)
        self.logger.addHandler(ch)
    
    def log_api_call(
        self,
        api: str,
        model: str,
        tokens_in: int,
        tokens_out: int,
        cost: float,
        duration_ms: float,
    ):
        """Log an API call."""
        self.logger.info(
            f"API_CALL: {api}/{model} - "
            f"tokens:{tokens_in+tokens_out} cost:${cost:.4f} duration:{duration_ms:.0f}ms"
        )
    
    def log_search_start(self, problem: str, config: dict):
        """Log start of ToT search."""
        self.logger.info(f"SEARCH_START: {config}")
    
    def log_search_complete(self, result: dict):
        """Log completion of ToT search."""
        self.logger.info(
            f"SEARCH_COMPLETE: score={result.get('best_score', 0):.1f} "
            f"cost=${result.get('total_cost', 0):.2f} "
            f"calls={result.get('total_calls', 0)}"
        )
    
    def log_error(self, error_type: str, message: str, recoverable: bool = True):
        """Log an error."""
        level = logging.WARNING if recoverable else logging.ERROR
        self.logger.log(level, f"ERROR: {error_type} - {message}")

print("✓ ProductionLogger loaded")
print("\nLogging categories:")
print("  - API calls (cost, tokens, latency)")
print("  - Search lifecycle (start, complete)")
print("  - Errors (recoverable vs critical)")
print("  - Cost warnings")
print("  - Performance metrics")

---
# Part 8: Summary and Quick Reference


In [ ]:
print("\n" + "="*70)
print("PHASE 3: LLM INTEGRATION SUMMARY")
print("="*70)

print("\n✓ Components Covered:")
components = {
    "1. API Setup": ["Claude (Anthropic)", "OpenAI (GPT-4, GPT-3.5)", "Environment variables & security"],
    "2. Prompting": ["Generation prompts", "Evaluation prompts", "Domain-specific templates"],
    "3. Orchestration": ["TreeOfThoughtsOrchestrator", "Thought tree management", "Cost tracking"],
    "4. Examples": ["Math problem solving", "Creative writing", "Code debugging"],
    "5. Best Practices": ["Batch processing", "Caching", "Token budgets", "Prompt optimization"],
    "6. Production": ["Rate limiting", "Fallback strategies", "Cost monitoring", "Logging"],
}

for category, items in components.items():
    print(f"\n{category}")
    for item in items:
        print(f"  ✓ {item}")

print("\n" + "="*70)
print("QUICK START EXAMPLE")
print("="*70)
print("""
# 1. Set up environment
from phase3_llm_integration import *
claud_client = ClaudeClient(api_key=os.getenv('ANTHROPIC_API_KEY'))

# 2. Create orchestrator
orchestrator = TreeOfThoughtsOrchestrator(
    problem="Your problem here",
    llm_client=claude_client,
    domain="math",
    max_depth=4,
    num_candidates=5,
    num_branches=2,
)

# 3. Run search
result = orchestrator.search(verbose=True)
print(f"Best path: {result['best_path']}")
print(f"Cost: ${result['total_cost']}")
""")

print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)
insights = [
    "Temperature 0.7-0.9 for generation (diversity)",
    "Temperature 0.0-0.2 for evaluation (consistency)",
    "Use cheaper models for evaluation to save costs",
    "Implement caching - identical prompts = 100% savings",
    "Fallback to simpler trees when APIs are overloaded",
    "Monitor costs per 1K tokens - aim for <0.05 cents",
    "Few-shot examples improve performance 10-50%",
    "Structured output (JSON) is more reliable",
]

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

print("\n" + "="*70)